In [1]:
from googleapiclient.discovery import build
import pandas as pd

def find_company_channels(api_key: str, company: str, max_results: int = 25):
    youtube = build("youtube", "v3", developerKey=api_key)

    # 1) Search YouTube for channels related to the company
    search_resp = youtube.search().list(
        part="snippet",
        q=company,
        type="channel",
        maxResults=max_results
    ).execute()

    channel_ids = [item["id"]["channelId"] for item in search_resp.get("items", [])]
    if not channel_ids:
        return pd.DataFrame()

    # 2) Fetch channel details (title, description, etc.)
    details_resp = youtube.channels().list(
        part="snippet",
        id=",".join(channel_ids)
    ).execute()

    rows = []
    for ch in details_resp.get("items", []):
        sn = ch["snippet"]
        rows.append({
            "channelId": ch["id"],
            "title": sn.get("title"),
            "description": sn.get("description"),
            "customUrl": sn.get("customUrl"),
            "publishedAt": sn.get("publishedAt"),
        })

    return pd.DataFrame(rows)


In [2]:
import sys
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("YOUTUBE_API_KEY")
# yt_client = YouTubeClient(api_key=api_key)
handle = "heineken"
find_company_channels(api_key, handle)

,channelId,title,description,customUrl,publishedAt
0,UCy-lH1kW6YDtO99Ic3Kig8g,Heineken NL,Social networking since 1873. Dit is het offic...,@heinekennl,2014-03-25T10:26:12Z
1,UCxGUQzCVgoEWE8P5JmYS-0g,Heineken Beahn,,@heinekenbeahn,2025-09-29T00:34:52.269185Z
2,UC4gYoef6CXHgAcH6aXKM2SA,Heineken Saint Lucia Ltd.,Celebrating 50 years of premium brews & unforg...,@heineken.saintlucia,2019-12-11T18:54:06.859462Z
3,UCKnjQiaumL_yTB0xmXCZ35A,Hennessy & Heineken,The twins are so loving and they mean the worl...,@henessyandheineken,2020-07-03T19:51:33.478093Z
4,UCxPeYqjR0q5l1kd39IVYFdw,HEINEKEN Ethiopia,,@heinekenethiopia,2024-01-29T11:18:57.023579Z
5,UCoXdHRKKrvDjOqe415bhCHQ,HEINEKEN KOREA,하이네켄 코리아는 설립 이후 매년 두자리 수의 성장률을 보이며 국내 수입맥주 시장을...,@heinekenkorea971,2019-05-21T08:50:09Z
6,UC_-HWQDxWNqEimlma9cJoIA,Heineken,This is an official Heineken® account. Our con...,@heineken,2005-09-07T19:02:01Z
7,UCwX3H8_w15ozNrEAxTOITaA,HEINEKEN INCONTRA,In Italia HEINEKEN è presente dal 1974 con i 4...,@heinekenincontra,2014-12-09T10:58:00Z
8,UCnbmmuVrPWzhnved8qrKeWA,Heineken Singapore,Welcome to the official Heineken Singapore You...,@heinekensingapore,2012-08-01T11:35:09Z
9,UCA4bkF4lKOMcMJgWodooIkw,Heineken Costa Rica,,@heinekencostarica6901,2022-06-04T04:15:44.035611Z


In [3]:
import re
import pandas as pd
from googleapiclient.discovery import build

_URL_RE = re.compile(r"(https?://[^\s]+)", re.IGNORECASE)

def _chunks(lst, n=50):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def _extract_urls(text: str):
    if not text:
        return []
    return _URL_RE.findall(text)

def _score_channel(row, brand_terms, official_handles=None, allow_region_handles=True,
                   min_subscribers=0, block_terms=None):
    """
    Returns (score:int, reasons:list[str])
    """
    official_handles = set(h.lower() for h in (official_handles or []))
    block_terms = set(t.lower() for t in (block_terms or []))
    brand_terms = [t.lower() for t in brand_terms]

    title = (row.get("title") or "").lower()
    desc  = (row.get("description") or "").lower()
    handle = (row.get("customUrl") or "").lower()
    subs = int(row.get("subscriberCount") or 0)
    vids = int(row.get("videoCount") or 0)

    score = 0
    reasons = []

    # 1) Hard positives: exact official handle(s)
    if handle and handle in official_handles:
        score += 100
        reasons.append("exact official handle match")

    # 2) Brand presence
    brand_in_title = any(bt in title for bt in brand_terms)
    brand_in_desc  = any(bt in desc for bt in brand_terms)
    if brand_in_title:
        score += 20; reasons.append("brand in title")
    if brand_in_desc:
        score += 10; reasons.append("brand in description")

    # 3) "Official" language
    if any(k in desc for k in ["official", "this is the official", "welcome to the official"]):
        score += 15; reasons.append("official wording in description")

    # 4) Region handle pattern (optional): @brand<country/region>
    # ex: @heinekenukcompany, @heinekensingapore, etc.
    if allow_region_handles and handle.startswith("@"):
        for bt in brand_terms:
            if handle.startswith("@" + bt) and handle not in official_handles:
                score += 8; reasons.append("region-style handle")
                break

    # 5) Scale by size/activeness
    if subs >= 100000: score += 10; reasons.append(">=100k subs")
    elif subs >= 10000: score += 6; reasons.append(">=10k subs")
    elif subs >= 1000: score += 3; reasons.append(">=1k subs")

    if vids >= 50: score += 5; reasons.append(">=50 videos")
    elif vids >= 10: score += 2; reasons.append(">=10 videos")

    # 6) URLs in description can be a good signal (optional heuristic)
    urls = _extract_urls(row.get("description") or "")
    if urls:
        score += 2; reasons.append("has URL(s) in description")

    # 7) Hard negatives: block terms
    # (tune this list per your dataset; keep it conservative)
    neg_hit = [t for t in block_terms if (t in title or t in desc or t in handle)]
    if neg_hit:
        score -= 40
        reasons.append(f"blocked terms: {', '.join(sorted(set(neg_hit)))}")

    # 8) Low-subs guardrail (optional)
    if min_subscribers and subs < min_subscribers:
        score -= 8
        reasons.append(f"subs < {min_subscribers}")

    return score, reasons

def filter_company_channels(api_key: str,
                            channels_df: pd.DataFrame,
                            brand_terms,
                            official_handles=None,
                            min_subscribers=500,
                            score_threshold=20,
                            block_terms=None):
    """
    Input: df from find_company_channels (channelId/title/description/customUrl/publishedAt)
    Output: df enriched + scored + filtered
    """
    if channels_df is None or channels_df.empty:
        return pd.DataFrame()

    youtube = build("youtube", "v3", developerKey=api_key)

    channel_ids = [c for c in channels_df["channelId"].dropna().unique().tolist()]
    enriched_rows = []

    # Enrich with stats + brandingSettings (keywords/country) + topicDetails
    for batch in _chunks(channel_ids, n=50):
        resp = youtube.channels().list(
            part="snippet,statistics,brandingSettings,topicDetails",
            id=",".join(batch)
        ).execute()

        for ch in resp.get("items", []):
            sn = ch.get("snippet", {}) or {}
            st = ch.get("statistics", {}) or {}
            bs = (ch.get("brandingSettings", {}) or {}).get("channel", {}) or {}
            td = ch.get("topicDetails", {}) or {}

            enriched_rows.append({
                "channelId": ch.get("id"),
                "title": sn.get("title"),
                "description": sn.get("description"),
                "customUrl": sn.get("customUrl"),
                "publishedAt": sn.get("publishedAt"),
                "country": sn.get("country") or bs.get("country"),
                "keywords": bs.get("keywords"),
                "subscriberCount": st.get("subscriberCount"),
                "videoCount": st.get("videoCount"),
                "viewCount": st.get("viewCount"),
                "topicCategories": (td.get("topicCategories") or []),
            })

    df = pd.DataFrame(enriched_rows)

    # Score + reasons
    scores = []
    reasons_col = []
    for _, row in df.iterrows():
        s, r = _score_channel(
            row,
            brand_terms=brand_terms,
            official_handles=official_handles,
            allow_region_handles=True,
            min_subscribers=min_subscribers,
            block_terms=block_terms
        )
        scores.append(s)
        reasons_col.append("; ".join(r))

    df["score"] = scores
    df["reasons"] = reasons_col

    # Filter
    kept = df[df["score"] >= score_threshold].sort_values("score", ascending=False).reset_index(drop=True)
    return kept


In [17]:
df_raw = find_company_channels(api_key, "@Cocacola", max_results=25)

df_clean = filter_company_channels(
    api_key=api_key,
    channels_df=df_raw,
    brand_terms=["heineken"],
    official_handles=["@heineken"],     # strong anchor
    min_subscribers=500,                # tune
    score_threshold=20,                 # tune
    block_terms=["xbox", "gaming", "account", "family", "doctor"]  # tune
)

df_clean[["channelId","title","customUrl","subscriberCount","videoCount","score","reasons"]]


,channelId,title,customUrl,subscriberCount,videoCount,score,reasons
0,UCBDwFwWPOXR1Ipxw6J9OOOg,Coca-Cola Canada,@openhappiness,15400,23,23,official wording in description; >=10k subs; >...


In [ ]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv

from googleapiclient.discovery import build

def get_featured_channels(api_key: str, channel_id: str) -> list[str]:
    yt = build("youtube", "v3", developerKey=api_key)

    resp = yt.channelSections().list(
        part="snippet,contentDetails",
        channelId=channel_id
    ).execute()

    featured = []
    for sec in resp.get("items", []):
        sec_type = sec.get("snippet", {}).get("type")

        # In actual responses, the value is camelCase: "multipleChannels" :contentReference[oaicite:2]{index=2}
        if sec_type in ("multiplechannels", "multiple_channels"):  # keep both just in case
            featured.extend(sec.get("contentDetails", {}).get("channels", []) or [])

    # de-dupe, preserve order
    return list(dict.fromkeys(featured))

# Load API key from .env if not already defined
load_dotenv()
api_key = os.getenv("YOUTUBE_API_KEY", None)
if api_key is None:
    raise ValueError("YOUTUBE_API_KEY not found in environment.")

# If channel ID for @heineken needs to be looked up:
from youtube_setup import YouTubeClient
yt_client = YouTubeClient(api_key=api_key)
heineken_channel_id = yt_client.get_channel_id("@Coca-Cola")
print("channel id:", heineken_channel_id)

featured_channels = get_featured_channels(api_key, heineken_channel_id)
print(featured_channels)


channel id: UCosXctaTYxN4YPIvI5Fpcrw
{'kind': 'youtube#channelSection', 'etag': 'Nn2-6sMEhHnhNpHdbUlfrN5Q8gA', 'id': 'UCosXctaTYxN4YPIvI5Fpcrw.jNQXAC9IVRw', 'snippet': {'type': 'channelsectiontypeundefined', 'channelId': 'UCosXctaTYxN4YPIvI5Fpcrw', 'position': 0}}
{'kind': 'youtube#channelSection', 'etag': 'w39RHWvlW4LOkVWWbARXl-cErHA', 'id': 'UCosXctaTYxN4YPIvI5Fpcrw.LeAltgu_pbM', 'snippet': {'type': 'popularuploads', 'channelId': 'UCosXctaTYxN4YPIvI5Fpcrw', 'position': 1}}
{'kind': 'youtube#channelSection', 'etag': 'W8ToIxT6RMrG0G0hz36-wF8k00A', 'id': 'UCosXctaTYxN4YPIvI5Fpcrw.nGzAI5pLbMY', 'snippet': {'type': 'channelsectiontypeundefined', 'channelId': 'UCosXctaTYxN4YPIvI5Fpcrw', 'position': 2}}
{'kind': 'youtube#channelSection', 'etag': 'TNaV57110DuPRWteBbogjGCSdPc', 'id': 'UCosXctaTYxN4YPIvI5Fpcrw.__WLXNpu6u8', 'snippet': {'type': 'multiplechannels', 'channelId': 'UCosXctaTYxN4YPIvI5Fpcrw', 'title': 'World of Coca-Cola', 'position': 3}, 'contentDetails': {'channels': ['UC26zJf6LAJ

In [29]:
from googleapiclient.discovery import build

def _chunks(xs, n=50):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def get_featured_channel_ids(api_key: str, channel_id: str) -> list[str]:
    yt = build("youtube", "v3", developerKey=api_key)

    resp = yt.channelSections().list(
        part="snippet,contentDetails",
        channelId=channel_id
    ).execute()

    featured = []
    for sec in resp.get("items", []):
        sec_type = (sec.get("snippet", {}).get("type") or "")
        norm = sec_type.replace("_", "").lower()

        # Works with "multiplechannels" and "multipleChannels" (and underscores)
        if norm == "multiplechannels":
            featured.extend(sec.get("contentDetails", {}).get("channels", []) or [])

    # de-dupe, preserve order
    return list(dict.fromkeys(featured))

def get_channel_id_and_handle(api_key: str, channel_ids: list[str]) -> list[dict]:
    """
    Returns list of dicts:
      {"channelId": "...", "handle": "@...", "title": "...", "customUrl": "..."}
    handle is verified (not guessed).
    """
    yt = build("youtube", "v3", developerKey=api_key)

    # preserve input order
    order = {cid: i for i, cid in enumerate(channel_ids)}
    results = []

    for chunk in _chunks([c for c in channel_ids if c], 50):
        resp = yt.channels().list(
            part="snippet",
            id=",".join(chunk)
        ).execute()

        for it in resp.get("items", []):
            cid = it["id"]
            sn = it.get("snippet", {}) or {}
            title = sn.get("title")
            custom = sn.get("customUrl")

            handle = None
            if isinstance(custom, str) and custom:
                if custom.startswith("@"):
                    handle = custom
                else:
                    # Verify whether "@{custom}" is the channel's handle
                    try:
                        vr = yt.channels().list(part="id", forHandle="@" + custom).execute()
                        items = vr.get("items", [])
                        if items and items[0].get("id") == cid:
                            handle = "@" + custom
                    except Exception:
                        pass

            results.append({
                "channelId": cid,
                "handle": handle,
                "title": title,
                "customUrl": custom,
            })

    results.sort(key=lambda x: order.get(x["channelId"], 10**9))
    return results

# ---- example usage ----
root_channel_id = yt_client.get_channel_id("@CocaCola")  # your existing function
featured_ids = get_featured_channel_ids(api_key, root_channel_id)
featured = get_channel_id_and_handle(api_key, featured_ids)
print(featured)


[{'channelId': 'UC26zJf6LAJpBb8thyvsvwyw', 'handle': '@cokestudioglobal', 'title': 'Coke Studio', 'customUrl': '@cokestudioglobal'}, {'channelId': 'UC5JBB_E5mzPEbDupD-6fA4A', 'handle': '@thecoca-colacompany', 'title': 'The Coca-Cola Co.', 'customUrl': '@thecoca-colacompany'}, {'channelId': 'UCKGaW3UnqvVM4D7iLF34zaA', 'handle': '@worldofcocacola', 'title': 'World of Coca-Cola', 'customUrl': '@worldofcocacola'}]


In [33]:
import re
from collections import deque, defaultdict
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

CHANNEL_ID_RE = re.compile(r"^UC[\w-]{22}$")

def _yt(api_key: str):
    return build("youtube", "v3", developerKey=api_key)

def _chunks(xs, n=50):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def resolve_channel_id(api_key: str, seed: str) -> str | None:
    yt = _yt(api_key)
    seed = (seed or "").strip()

    # 1) Already a channelId?
    if CHANNEL_ID_RE.match(seed):
        try:
            r = yt.channels().list(part="id", id=seed).execute()
            return r["items"][0]["id"] if r.get("items") else None
        except Exception:
            return None

    # 2) Handle-like: try forHandle, but DO NOT rely on it
    if seed.startswith("@") and " " not in seed:
        # try forHandle first (if supported)
        try:
            r = yt.channels().list(part="id", forHandle=seed).execute()
            if r.get("items"):
                return r["items"][0]["id"]
        except Exception:
            pass  # fall through to search

        # search fallback (handles often resolve fine via search)
        q = seed[1:]
    else:
        q = seed

    # 3) Search fallback (robust parsing: id.channelId)
    try:
        s = yt.search().list(part="id,snippet", q=q, type="channel", maxResults=5).execute()
        for item in s.get("items", []):
            cid = (item.get("id") or {}).get("channelId")
            if cid:
                return cid
    except Exception:
        return None

    return None

def get_featured_channel_ids(api_key: str, channel_id: str) -> list[str]:
    yt = _yt(api_key)
    resp = yt.channelSections().list(
        part="snippet,contentDetails",
        channelId=channel_id
    ).execute()

    featured = []
    for sec in resp.get("items", []):
        sec_type = (sec.get("snippet", {}).get("type") or "")
        norm = sec_type.replace("_", "").lower()
        if norm == "multiplechannels":  # matches your debug output
            featured.extend(sec.get("contentDetails", {}).get("channels", []) or [])

    return list(dict.fromkeys(featured))

def get_channels_meta_with_handle(api_key: str, channel_ids: list[str]) -> dict:
    """
    Returns {channelId: {"title":..., "customUrl":..., "handle":...}}
    handle = customUrl if it's already like '@xyz', else None.
    (This is the only reliable "handle-like" field you can get from Data API v3.)
    """
    yt = _yt(api_key)
    meta = {}

    ids = [c for c in channel_ids if c]
    for chunk in _chunks(ids, 50):
        r = yt.channels().list(part="snippet", id=",".join(chunk)).execute()
        for it in r.get("items", []):
            cid = it["id"]
            sn = it.get("snippet", {}) or {}
            custom = sn.get("customUrl")
            handle = custom if isinstance(custom, str) and custom.startswith("@") else None
            meta[cid] = {
                "title": sn.get("title"),
                "customUrl": custom,
                "handle": handle,
            }
    return meta

def bfs_featured_graph(api_key: str, seeds, max_depth: int = 2, max_nodes: int = 300):
    if isinstance(seeds, str):
        seeds = [seeds]

    seed_ids = []
    for s in seeds:
        cid = resolve_channel_id(api_key, s)
        if cid:
            seed_ids.append(cid)

    if not seed_ids:
        raise ValueError(f"No valid seed channels resolved from: {seeds}")

    visited = set(seed_ids)
    depth_of = {cid: 0 for cid in seed_ids}
    q = deque((cid, 0) for cid in seed_ids)

    edges = defaultdict(list)
    new_by_depth = defaultdict(list)

    while q and len(visited) < max_nodes:
        cur, d = q.popleft()
        if d >= max_depth:
            continue

        try:
            children = get_featured_channel_ids(api_key, cur)
        except HttpError:
            continue

        edges[cur] = children
        nd = d + 1

        for ch in children:
            if ch not in visited:
                visited.add(ch)
                depth_of[ch] = nd
                new_by_depth[nd].append(ch)
                q.append((ch, nd))
                if len(visited) >= max_nodes:
                    break

    nodes_meta = get_channels_meta_with_handle(api_key, list(visited))
    return nodes_meta, dict(edges), dict(new_by_depth), depth_of

nodes, edges, new_by_depth, depth_of = bfs_featured_graph(
    api_key,
    seeds="@CocaCola",   # or "Coca-Cola", or UC...
    max_depth=3,
    max_nodes=500
)
print("nodes:", len(nodes))
print("depth1:", len(new_by_depth.get(1, [])))
print(nodes)



nodes: 5
depth1: 3
{'UC5JBB_E5mzPEbDupD-6fA4A': {'title': 'The Coca-Cola Co.', 'customUrl': '@thecoca-colacompany', 'handle': '@thecoca-colacompany'}, 'UCosXctaTYxN4YPIvI5Fpcrw': {'title': 'Coca-Cola', 'customUrl': '@coca-cola', 'handle': '@coca-cola'}, 'UCdxTX9FGFPrW16WQPEXJJlA': {'title': 'CokeConversations', 'customUrl': '@cokeconversations', 'handle': '@cokeconversations'}, 'UC26zJf6LAJpBb8thyvsvwyw': {'title': 'Coke Studio', 'customUrl': '@cokestudioglobal', 'handle': '@cokestudioglobal'}, 'UCKGaW3UnqvVM4D7iLF34zaA': {'title': 'World of Coca-Cola', 'customUrl': '@worldofcocacola', 'handle': '@worldofcocacola'}}


In [13]:
import os
import numpy as np
import pandas as pd
from openai import OpenAI

def _chunks(lst, n=100):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def _l2_normalize(v: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / (n + 1e-12)

def build_channel_profile_text(row: dict) -> str:
    # Keep it brand-heavy + stable
    parts = [
        f"TITLE: {row.get('title','')}",
        f"HANDLE: {row.get('customUrl','')}",
        f"DESCRIPTION: {row.get('description','')}",
        f"KEYWORDS: {row.get('keywords','') or ''}",
        f"COUNTRY: {row.get('country','') or ''}",
        "TOPIC_CATEGORIES: " + " ".join(row.get("topicCategories") or []),
    ]
    return "\n".join([p for p in parts if p and p.strip()])

def openai_embed_texts(
    texts: list[str],
    model: str = "text-embedding-3-small",
    dimensions: int | None = None,
    batch_size: int = 128,
    max_chars: int = 12000,   # crude guardrail; avoids huge video descriptions
):
    """
    Returns: np.ndarray shape (len(texts), dim)
    """
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    # cheap truncation (token-based is better, but this is usually fine)
    cleaned = [(t or "")[:max_chars] for t in texts]

    embs = []
    for batch in _chunks(cleaned, batch_size):
        kwargs = dict(model=model, input=batch)
        # `dimensions` only supported on text-embedding-3* models. :contentReference[oaicite:1]{index=1}
        if dimensions is not None:
            kwargs["dimensions"] = dimensions

        resp = client.embeddings.create(**kwargs)  # encoding_format defaults to float :contentReference[oaicite:2]{index=2}
        embs.extend([d.embedding for d in resp.data])

    return np.asarray(embs, dtype=np.float32)

def filter_channels_by_seed_embedding_openai(
    seed_row: dict,
    candidates_df: pd.DataFrame,
    model: str = "text-embedding-3-small",
    dimensions: int | None = None,
    sim_threshold: float = 0.50,
    top_k: int = 50,
):
    """
    seed_row: dict for the official channel (already enriched)
    candidates_df: enriched df with candidate channels
    """
    seed_text = build_channel_profile_text(seed_row)
    cand_texts = [build_channel_profile_text(r) for r in candidates_df.to_dict("records")]

    seed_emb = openai_embed_texts([seed_text], model=model, dimensions=dimensions)[0]
    cand_embs = openai_embed_texts(cand_texts, model=model, dimensions=dimensions)

    seed_emb = _l2_normalize(seed_emb)
    cand_embs = np.vstack([_l2_normalize(v) for v in cand_embs])

    sims = cand_embs @ seed_emb  # cosine because we normalized
    out = candidates_df.copy()
    out["sim_to_seed"] = sims

    out = out.sort_values("sim_to_seed", ascending=False)
    out = out[out["sim_to_seed"] >= sim_threshold].head(top_k).reset_index(drop=True)
    return out


In [14]:
# df_raw = find_company_channels(api_key, "heineken")
candidate_ids = df_raw["channelId"].dropna().unique().tolist()

# df_enriched = enrich_channels(YT_API_KEY, candidate_ids + [seed_id])  # your existing enrich
df_enriched = df_clean
seed_id = "UC_-HWQDxWNqEimlma9cJoIA"  # @heineken
seed_row = df_enriched[df_enriched["channelId"] == seed_id].iloc[0].to_dict()
cand_df  = df_enriched[df_enriched["channelId"] != seed_id].reset_index(drop=True)

df_match = filter_channels_by_seed_embedding_openai(
    seed_row=seed_row,
    candidates_df=cand_df,
    model="text-embedding-3-small",   # or text-embedding-3-large :contentReference[oaicite:3]{index=3}
    sim_threshold=0.50,               # if you get empty, drop to 0.40
    top_k=50,
)

df_match[["channelId","title","customUrl","subscriberCount","videoCount","sim_to_seed"]]


,channelId,title,customUrl,subscriberCount,videoCount,sim_to_seed
0,UCCG2Xo_VnkBDY0xMU5tZsVA,Heineken,@heineken-b2o,0,0,0.829462
1,UCHl25QQ8frtx-okPJJLWIOg,@Heineken,@heinekencod,2,1,0.824140
2,UCMiUSAvuP9k-K2LGGDWMmJQ,Heineken NZ,@heinekennz,376,10,0.824108
3,UCoiapnTMUfKYwLXcrrC3iPA,Heineken Ireland,@heinekenire,2670,186,0.811734
4,UCnbmmuVrPWzhnved8qrKeWA,Heineken Singapore,@heinekensingapore,1710,48,0.787375
5,UCxGUQzCVgoEWE8P5JmYS-0g,Heineken Beahn,@heinekenbeahn,5130,119,0.781261
6,UCOfv9uRTJp4nZ3sGSvR3xpQ,Heineken Vietnam,@heinekenvietnam,71900,293,0.773934
7,UC0VHMRpkk1cYaO8jkRQ23pg,Heineken Hungary,@heinekenhungary,512,93,0.759238
8,UCNigfD8Sfu0Qe6zil52A_2A,@HEINEKEN Career Brewer,@heinekentwcareerbrewer,34,12,0.757764
9,UCxPeYqjR0q5l1kd39IVYFdw,HEINEKEN Ethiopia,@heinekenethiopia,161,13,0.745854
